<a href="https://colab.research.google.com/github/SaraBatistaPereira/ORI/blob/main/exercicio1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Exercício 1 - Processamento de Texto e TF-IDF

**Grupo:**
* Eduardo de Oliveira Targa Santos
* João Paulo Pinto Alves
* Leonardo Balestere Machado
* Luis Gustavo Mariani Nogueira
* Mateus Granjeia Santos
* Matheus Felipe Ferreira Pereira
* Sara Batista
```

```markdown
## 1. Setup e Criação dos Documentos de Teste
Para testar nosso programa, vamos criar um diretório temporário `./documentos_teste/` e preenchê-lo com 4 arquivos de texto contendo mais de 100 palavras cada.
```

In [1]:
import os
import math
import string
import unicodedata
import pandas as pd

# Criando pasta para os documentos de teste
os.makedirs('./documentos_teste', exist_ok=True)

# Definindo 4 textos com pelo menos 100 palavras cada
doc1 = """A inteligência artificial é um campo da ciência da computação que se dedica ao estudo e criação de mecanismos ou dispositivos que simulem a inteligência humana. Esse ramo tecnológico foca no desenvolvimento de algoritmos que permitem a máquinas aprender de maneira autônoma com base em grandes volumes de dados disponíveis. Diversas ferramentas de IA hoje auxiliam em diagnósticos médicos detalhados, automação de processos industriais complexos e até na tradução simultânea de idiomas. Com o avanço das redes neurais profundas, novas oportunidades surgiram no mercado, transformando completamente o cotidiano da sociedade moderna e exigindo novos cuidados éticos importantes para o futuro."""

doc2 = """O desenvolvimento de software moderno exige metodologias ágeis que facilitem a entrega contínua de valor ao cliente final de forma rápida e segura. Equipes de engenharia de software utilizam práticas como integração contínua e implantação contínua para minimizar erros de código em produção de modo eficiente. Além disso, o uso de testes automatizados e revisões de código por pares garante que a qualidade do produto final seja mantida em níveis excelentes. A colaboração estreita entre desenvolvedores, designers e gerentes de produto é fundamental para o sucesso de qualquer projeto tecnológico de grande ou pequeno porte na atualidade."""

doc3 = """A astronomia é uma das ciências mais antigas do mundo e estuda corpos celestes como estrelas, planetas, cometas, nebulosas e galáxias inteiras no universo. Cientistas utilizam telescópios espaciais avançados localizados fora da atmosfera da Terra para capturar imagens impressionantes de galáxias distantes e analisar a luz emitida por estrelas antigas. A busca por exoplanetas habitáveis em outros sistemas solares tem se intensificado nos últimos anos com o avanço de novos métodos de detecção e espectroscopia de alta precisão. Descobrir vida extraterrestre ou compreender a expansão acelerada do cosmos continuam sendo os maiores e mais intrigantes desafios da ciência atual."""

doc4 = """A prática regular de atividades físicas aliada a uma alimentação equilibrada e saudável é essencial para a manutenção da saúde física e mental a longo prazo. Estudos demonstram que caminhar, correr ou praticar esportes coletivos reduz significativamente os riscos de doenças cardiovasculares crônicas e melhora o humor geral do indivíduo de forma natural. Além dos benefícios biológicos evidentes, o esporte promove a socialização ativa, a disciplina individual e o alívio do estresse acumulado na rotina de trabalho diária. Profissionais da saúde recomendam pelo menos trinta minutos de exercícios diários moderados para garantir uma vida longa e cheia de bem-estar contínuo."""

# Salvando os arquivos na pasta criada
textos = [doc1, doc2, doc3, doc4]
for i, texto in enumerate(textos, start=1):
    with open(f'./documentos_teste/documento_{i}.txt', 'w', encoding='utf-8') as f:
        f.write(texto)

print("Documentos de teste gerados com sucesso na pasta './documentos_teste'!")
for i, texto in enumerate(textos, start=1):
    print(f"Documento {i}: {len(texto.split())} palavras.")

Documentos de teste gerados com sucesso na pasta './documentos_teste'!
Documento 1: 100 palavras.
Documento 2: 97 palavras.
Documento 3: 99 palavras.
Documento 4: 100 palavras.


```markdown
## 2. Funções Modulares
Nesta etapa, implementaremos as funções de pré-processamento de texto para remover acentos, pontuação e converter para letras minúsculas. Em seguida, implementaremos as funções para criar o vocabulário, calcular o TF, o IDF e o TF-IDF final.
```

In [2]:
def limpar_e_tokenizar(texto):
    """
    Remove acentos, pontuação, quebras de linha e retorna uma lista de palavras minúsculas.
    """
    # Normaliza acentos utilizando unicodedata
    texto_normalizado = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')
    # Converte para minúsculas
    texto_minusculo = texto_normalizado.lower()
    # Remove pontuação substituindo por espaço
    tabela_pontuacao = str.maketrans(string.punctuation, ' ' * len(string.punctuation))
    texto_sem_pontuacao = texto_minusculo.translate(tabela_pontuacao)
    # Tokeniza as palavras
    palavras = texto_sem_pontuacao.split()
    return palavras

def criar_vocabulario(diretorio):
    """
    Lê todos os arquivos do diretório, limpa os textos e retorna o vocabulário ordenado alfabeticamente.
    """
    vocabulario = set()
    documentos = {}

    # Lista todos os arquivos .txt no diretório
    arquivos = [f for f in os.listdir(diretorio) if f.endswith('.txt')]

    for arquivo in arquivos:
        caminho_completo = os.path.join(diretorio, arquivo)
        with open(caminho_completo, 'r', encoding='utf-8') as f:
            conteudo = f.read()
            palavras = limpar_e_tokenizar(conteudo)
            documentos[arquivo] = palavras
            vocabulario.update(palavras)

    return sorted(list(vocabulario)), documentos

def salvar_vocabulario(vocabulario, caminho_saida):
    """
    Salva o vocabulário ordenado em um arquivo de texto.
    """
    with open(caminho_saida, 'w', encoding='utf-8') as f:
        for termo in vocabulario:
            f.write(termo + '\n')
    print(f"Vocabulário com {len(vocabulario)} termos salvo com sucesso em '{caminho_saida}'!")

In [3]:
def calcular_tf(documento_palavras):
    """
    Calcula o Term Frequency (TF) para cada termo no documento.
    Utiliza a frequência absoluta das palavras.
    """
    tf_dict = {}
    for palavra in documento_palavras:
        tf_dict[palavra] = tf_dict.get(palavra, 0) + 1
    return tf_dict

def calcular_idf(documentos, vocabulario):
    """
    Calcula o Inverse Document Frequency (IDF) de cada palavra do vocabulário.
    Fórmula utilizada: IDF = log_10( N / DF_t )
    """
    idf_dict = {}
    N = len(documentos)

    for termo in vocabulario:
        # Conta em quantos documentos o termo aparece
        df_t = sum(1 for doc_palavras in documentos.values() if termo in doc_palavras)
        # Evita divisão por zero caso o termo não exista nos documentos
        if df_t > 0:
            idf_dict[termo] = math.log10(N / df_t)
        else:
            idf_dict[termo] = 0.0

    return idf_dict

def calcular_tfidf(documentos, vocabulario, idf_dict):
    """
    Calcula a matriz de pesos TF-IDF final para cada documento e termo.
    """
    tfidf_resultados = {}

    for nome_doc, palavras in documentos.items():
        tf_dict = calcular_tf(palavras)
        tfidf_resultados[nome_doc] = {}
        for termo in vocabulario:
            tf_val = tf_dict.get(termo, 0)
            idf_val = idf_dict.get(termo, 0)
            tfidf_resultados[nome_doc][termo] = tf_val * idf_val

    return tfidf_resultados

```markdown
## 3. Execução Principal
Vamos integrar todas as etapas, salvando o arquivo do vocabulário e exibindo os resultados calculados para cada documento.
```

In [4]:
diretorio_entrada = './documentos_teste'
arquivo_vocabulario = 'vocabulario.txt'

# 1. Criar vocabulário e ler documentos
vocabulario, documentos = criar_vocabulario(diretorio_entrada)

# 2. Salvar vocabulário gerado em ordem alfabética
salvar_vocabulario(vocabulario, arquivo_vocabulario)

# 3. Calcular IDF para cada termo na coleção
idfs = calcular_idf(documentos, vocabulario)

# 4. Calcular pesos TF-IDF
tfidf_matriz = calcular_tfidf(documentos, vocabulario, idfs)

# 5. Exibir os resultados em um DataFrame do Pandas para fácil visualização
df_resultados = pd.DataFrame(tfidf_matriz)

print("\n--- Visualização da Matriz TF-IDF (Primeiras 30 linhas do vocabulário) ---")
display(df_resultados.head(30))

# Verificação simples: exibir os termos com maiores pesos em cada documento
print("\n--- Termos com maiores pesos TF-IDF por documento ---")
for doc, pesos in tfidf_matriz.items():
    termos_ordenados = sorted(pesos.items(), key=lambda item: item[1], reverse=True)
    print(f"\nDocumento: {doc}")
    for termo, peso in termos_ordenados[:5]:
        if peso > 0:
            print(f"  - '{termo}': {peso:.4f}")

Vocabulário com 261 termos salvo com sucesso em 'vocabulario.txt'!

--- Visualização da Matriz TF-IDF (Primeiras 30 linhas do vocabulário) ---


,documento_4.txt,documento_3.txt,documento_1.txt,documento_2.txt
a,0.00000,0.00000,0.00000,0.00000
acelerada,0.00000,0.60206,0.00000,0.00000
acumulado,0.60206,0.00000,0.00000,0.00000
ageis,0.00000,0.00000,0.00000,0.60206
alem,0.30103,0.00000,0.00000,0.30103
algoritmos,0.00000,0.00000,0.60206,0.00000
aliada,0.60206,0.00000,0.00000,0.00000
alimentacao,0.60206,0.00000,0.00000,0.00000
alivio,0.60206,0.00000,0.00000,0.00000
alta,0.00000,0.60206,0.00000,0.00000



--- Termos com maiores pesos TF-IDF por documento ---

Documento: documento_4.txt
  - 'saude': 1.2041
  - 'acumulado': 0.6021
  - 'aliada': 0.6021
  - 'alimentacao': 0.6021
  - 'alivio': 0.6021

Documento: documento_3.txt
  - 'antigas': 1.2041
  - 'estrelas': 1.2041
  - 'galaxias': 1.2041
  - 'mais': 1.2041
  - 'acelerada': 0.6021

Documento: documento_1.txt
  - 'inteligencia': 1.2041
  - 'algoritmos': 0.6021
  - 'aprender': 0.6021
  - 'artificial': 0.6021
  - 'ate': 0.6021

Documento: documento_2.txt
  - 'continua': 1.8062
  - 'codigo': 1.2041
  - 'final': 1.2041
  - 'produto': 1.2041
  - 'software': 1.2041


In [6]:
# 1. Gerando o DataFrame de TF (Term Frequency)
tfs_matriz = {}
for nome_doc, palavras in documentos.items():
    tf_dict = calcular_tf(palavras)
    tfs_matriz[nome_doc] = {termo: tf_dict.get(termo, 0) for termo in vocabulario}

df_tf = pd.DataFrame(tfs_matriz)

# 2. Gerando o DataFrame de IDF (Inverse Document Frequency)
df_idf = pd.DataFrame.from_dict(idfs, orient='index', columns=['IDF'])

# 3. Gerando o DataFrame de TF-IDF completo
df_tfidf = pd.DataFrame(tfidf_matriz)

# Criando uma função para destacar os valores maiores que zero para melhor leitura
def destacar_valores(val):
    if val == 0:
        return 'background-color: #ffffff; color: #b5b5b5; font-style: italic;'
    return 'background-color: #e8f0fe; color: #1a73e8; font-weight: bold;'

print("\n=========================================================================")
print("1. TABELA DE FREQUÊNCIA DE TERMOS (TF) - Primeiras 25 linhas")
print("=========================================================================")
# Usando .map em vez de .applymap para evitar o aviso de descontinuação
display(df_tf.head(25).style.map(destacar_valores))

print("\n=========================================================================")
print("2. TABELA DE FREQUÊNCIA INVERSA DE DOCUMENTOS (IDF) - Primeiras 25 linhas")
print("=========================================================================")
# Exibindo o IDF formatado com 4 casas decimais e gradiente sutil de relevância
display(df_idf.head(25).style.format('{:.4f}').background_gradient(cmap='Blues'))

print("\n=========================================================================")
print("3. MATRIZ DE PESOS TF-IDF COMPLETA - Primeiras 25 linhas")
print("=========================================================================")
# Formatando o TF-IDF com 4 casas decimais e aplicando um gradiente elegante de azul e verde (YlGnBu)
display(df_tfidf.head(25).style.format('{:.4f}').background_gradient(cmap='YlGnBu', axis=None))



1. TABELA DE FREQUÊNCIA DE TERMOS (TF) - Primeiras 25 linhas


,documento_4.txt,documento_3.txt,documento_1.txt,documento_2.txt
a,6,4,3,3
acelerada,0,1,0,0
acumulado,1,0,0,0
ageis,0,0,0,1
alem,1,0,0,1
algoritmos,0,0,1,0
aliada,1,0,0,0
alimentacao,1,0,0,0
alivio,1,0,0,0
alta,0,1,0,0



2. TABELA DE FREQUÊNCIA INVERSA DE DOCUMENTOS (IDF) - Primeiras 25 linhas


,IDF
a,0.0000
acelerada,0.6021
acumulado,0.6021
ageis,0.6021
alem,0.3010
algoritmos,0.6021
aliada,0.6021
alimentacao,0.6021
alivio,0.6021
alta,0.6021



3. MATRIZ DE PESOS TF-IDF COMPLETA - Primeiras 25 linhas


,documento_4.txt,documento_3.txt,documento_1.txt,documento_2.txt
a,0.0000,0.0000,0.0000,0.0000
acelerada,0.0000,0.6021,0.0000,0.0000
acumulado,0.6021,0.0000,0.0000,0.0000
ageis,0.0000,0.0000,0.0000,0.6021
alem,0.3010,0.0000,0.0000,0.3010
algoritmos,0.0000,0.0000,0.6021,0.0000
aliada,0.6021,0.0000,0.0000,0.0000
alimentacao,0.6021,0.0000,0.0000,0.0000
alivio,0.6021,0.0000,0.0000,0.0000
alta,0.0000,0.6021,0.0000,0.0000
